# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandera Faker
    print('Packages installed.')

# Data Quality Validation with Pandera

In social science research, data quality is paramount. Manual checks are prone to error. `pandera` allows you to define a **schema** for your data and automatically validate it, catching errors early.

This notebook covers:
1.  **Generating 'Dirty' Data**: Simulating real-world errors (typos, out-of-range values).
2.  **Defining a Schema**: specifying expected data types and constraints.
3.  **Validation**: Running checks and handling failures.
4.  **Cleaning & Re-validating**: Fixing the data to pass the schema.

In [ ]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, Check, DataFrameSchema
from faker import Faker
import random

fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

## 1. Generate 'Dirty' Synthetic Data

We'll simulate a survey dataset with some common issues:
-   **Age**: Should be 18-100, but we'll insert some negative numbers.
-   **Income**: Should be positive, but we'll insert a string "Unknown".
-   **Consent**: Should be Boolean (True/False), but we'll have missing values.

In [ ]:
def generate_dirty_data(n=100):
    data = []
    for _ in range(n):
        row = {
            'id': fake.uuid4(),
            'age': random.randint(18, 90),
            'income': random.randint(20000, 150000),
            'gender': random.choice(['Male', 'Female', 'Non-binary']),
            'consent': True
        }
        data.append(row)
    
    df = pd.DataFrame(data)
    
    # Introduce errors
    # 1. Negative Age
    df.loc[0, 'age'] = -5
    
    # 2. String in Income (numeric column)
    df = df.astype({'income': 'object'}) # Convert to object to allow string
    df.loc[1, 'income'] = "Unknown"
    
    # 3. Missing Consent
    df['consent'] = df['consent'].astype(object)
    df.loc[2, 'consent'] = None
    
    return df

df_dirty = generate_dirty_data()
print("Dirty Data Sample:")
print(df_dirty.head())

## 2. Define Validation Schema

We define what the data *should* look like.

In [ ]:
schema = DataFrameSchema({
    # Age must be int, between 18 and 100
    "age": Column(int, Check.in_range(18, 100)),
    
    # Income must be int (this will fail on "Unknown")
    "income": Column(int, Check.greater_than(0)),
    
    # Gender must be one of the specified categories
    "gender": Column(str, Check.isin(['Male', 'Female', 'Non-binary'])),
    
    # Consent cannot be null
    "consent": Column(bool, nullable=False)
})

## 3. Attempt Validation

Running this will raise a `SchemaError` explaining exactly what is wrong.

In [ ]:
try:
    schema.validate(df_dirty)
except pa.errors.SchemaErrors as err:
    print("Validation Failed!")
    print(err.failure_cases) # Shows exactly which rows/columns failed
except Exception as e:
    print(f"Validation Failed with error: {e}")

## 4. Cleaning the Data

Now we fix the identified issues.

In [ ]:
df_clean = df_dirty.copy()

# Fix Age: Replace negative with mean or drop. Let's drop for now.
df_clean = df_clean[df_clean['age'] >= 18]

# Fix Income: Coerce errors to NaN, then fill or drop
df_clean['income'] = pd.to_numeric(df_clean['income'], errors='coerce')
df_clean = df_clean.dropna(subset=['income'])
df_clean['income'] = df_clean['income'].astype(int)

# Fix Consent: Drop missing
df_clean = df_clean.dropna(subset=['consent'])
df_clean['consent'] = df_clean['consent'].astype(bool)

print(f"Rows remaining: {len(df_clean)}")

## 5. Re-Validation

Now the data should pass.

In [ ]:
validated_df = schema.validate(df_clean)
print("Validation Successful!")
print(validated_df.head())